In [1]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. LOAD & MERGE DATA
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
market_data = pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)

# Merge features + returns
data = state_data.join(market_data[["nifty_ret"]], how="inner")

print("Columns:", data.columns.tolist())
print(f"Dataset shape: {data.shape}\n")

# =============================================
# 2. ENVIRONMENT
# =============================================
class TradingEnv(gym.Env):
    
    def __init__(self, data, initial_balance=1000000):
        super(TradingEnv, self).__init__()
        
        self.data = data.reset_index(drop=True)
        self.initial_balance = initial_balance
        
        # FINAL FEATURE SET
        self.features = [
            "mom_20_norm",
            "vol_signal_norm",
            "trend_signal_norm",
            "dd_signal_norm",
            "vix_signal_norm",
            "breadth_signal_norm"
        ]
        
        # Safety checks
        missing = [col for col in self.features if col not in self.data.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        
        if "nifty_ret" not in self.data.columns:
            raise ValueError("Column 'nifty_ret' missing")
        
        # Action space: 0=Flat, 1=Long, 2=Short
        self.action_space = spaces.Discrete(3)
        
        # Observation space: features + position
        self.observation_space = spaces.Box(
            low=-5,
            high=5,
            shape=(len(self.features) + 1,),
            dtype=np.float32
        )
        
        self.max_steps = len(self.data) - 1
        self.reset()
    
    # =============================================
    # RESET
    # =============================================
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = self.initial_balance
        self.peak = self.initial_balance
        
        return self._get_observation()
    
    # =============================================
    # STATE
    # =============================================
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        
        obs = row[self.features].values.astype(np.float32)
        obs = np.append(obs, self.position)
        
        return obs
    
    # =============================================
    # STEP FUNCTION (FINAL)
    # =============================================
    def step(self, action):
        
        # Action → position
        new_position = {0: 0, 1: 1, 2: -1}[action]
        prev_position = self.position
        
        # Market return
        current_ret = self.data.iloc[self.current_step]["nifty_ret"]
        
        # Transaction cost
        cost = abs(new_position - prev_position) * 0.0005
        
        # Base return
        strategy_ret = prev_position * current_ret
        
        # Update balance
        self.balance *= (1 + strategy_ret - cost)
        
        # Drawdown tracking
        self.peak = max(self.peak, self.balance)
        drawdown = (self.balance / self.peak) - 1
        
        # =============================
        # REWARD FUNCTION (ADVANCED)
        # =============================
        
        # Risk penalty (volatility control)
        risk_penalty = 0.1 * (strategy_ret ** 2)
        
        # Drawdown penalty (capital protection)
        dd_penalty = 0.2 * abs(drawdown)
        
        # Final reward
        reward = strategy_ret - cost - risk_penalty - dd_penalty
        
        # Update position
        self.position = new_position
        
        # Move forward
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {
            "balance": self.balance,
            "position": self.position,
            "drawdown": drawdown
        }
    
    # =============================================
    # DEBUG
    # =============================================
    def render(self):
        print(f"Step: {self.current_step}, Position: {self.position}, Balance: {self.balance:.2f}")

# =============================================
# 3. CREATE ENVIRONMENT
# =============================================
env = TradingEnv(data)

print("Environment created successfully")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")
print(f"Total steps: {env.max_steps}")

# =============================================
# 4. TEST RUN
# =============================================
print("\n--- Random Agent Test ---")

obs = env.reset()
total_reward = 0

for step in range(100):
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    
    total_reward += reward
    
    if step % 20 == 0:
        print(f"Step {step}: Reward={reward:.5f}, Balance={info['balance']:.2f}")
    
    if done:
        break

print("\nTest Completed")
print(f"Total Reward: {total_reward:.5f}")
print(f"Final Balance: {info['balance']:.2f}")

Columns: ['mom_20_norm', 'vol_signal_norm', 'trend_signal_norm', 'dd_signal_norm', 'vix_signal_norm', 'breadth_signal_norm', 'nifty_ret']
Dataset shape: (1529, 7)

Environment created successfully
Observation shape: (7,)
Action space: Discrete(3)
Total steps: 1528

--- Random Agent Test ---
Step 0: Reward=-0.00060, Balance=999500.00
Step 20: Reward=0.00122, Balance=972621.93
Step 40: Reward=0.01701, Balance=897926.45
Step 60: Reward=-0.04486, Balance=742921.85
Step 80: Reward=-0.05205, Balance=739732.96

Test Completed
Total Reward: -3.45075
Final Balance: 769719.59
